# Instructions

- This homework assignment is worth **74** points. 
- Please strive for clarity and organization.
- **AI Usage**:
   - AI use is strictly **prohibited** for conceptual exercises.
   - For applied (Python) exercises, you may only use AI for **debugging**. Do not use AI to generate answers.
- **Due Date: October 2, 2026, by 11:59 PM.**

# Exercise 1 

In this exercise, we will work with the `winemag-data-130k-v2.csv` data file. This file contains information about wine. The goal is to explore this data using the NLP techniques that we have learnt. For more information about the data file, see this [link](https://www.kaggle.com/datasets/zynicide/wine-reviews/data).

### Exercise 1(a) (2 points)

Load the following libraries as follows:

```
import pandas as pd 
import numpy as np

import matplotlib.pyplot as plt
```

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Exercise 1(b) (2 points)

Read the `csv` file and create a data-frame called `wine_df`.

In [3]:
wine_df = pd.read_csv('DATA/winemag-data-130k-v2.csv')

### Exercise 1(c) (2 points)

Report the number of wine types in the `wine_df` data frame.

In [4]:
wine_df['variety'].nunique()

707

### Exercise 1(d) (2 points)

Report the number countries producing wine in the `wine_df` data frame.

In [5]:
wine_df['country'].nunique()

43

### Exercise 1(e) (3 points)

Using the `string` library, remove all the punctuation of the `description` column and store the results in another column called `description_clean`.

In [6]:
import string

wine_df['description_clean'] = wine_df['description'].str.translate(
    str.maketrans('', '', string.punctuation)
)

### Exercise 1(f) (10 points)

Using the `nltk` library, remove stopwords, pronouns, and the following words: `drink`, `now`, `wine`, `flavor`, and `flavors` from `description_clean` and store in another column called `'description_clean_1'`. 

In [7]:
import nltk
from nltk.corpus import stopwords
from nltk import pos_tag, word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download(['stopwords', 'punkt_tab', 'wordnet',
               'averaged_perceptron_tagger_eng'])

stop = set(stopwords.words('english'))
extra = {'drink', 'now', 'wine', 'flavor', 'flavors'}

def clean_text(t):
    toks = word_tokenize(str(t).lower())
    tagged = pos_tag(toks)
    return ' '.join(w for w, tag in tagged
                    if w not in stop and w not in extra
                    and tag not in ('PRP', 'PRP$', 'WP', 'WP$'))

wine_df['description_clean_1'] = wine_df['description_clean'].apply(clean_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


### Exercise 1(g) (10 points)

Perform Lemmatization of `description_clean_1` and store the results in another column called `description_clean_2` in the `wine_df` data frame.

In [8]:
lem = WordNetLemmatizer()
wine_df['description_clean_2'] = wine_df['description_clean_1'].apply(
    lambda t: ' '.join(lem.lemmatize(w) for w in t.split())
)

### Exercise 1(h) (5 points)

Based on the results from part 1(g), we see digits in some of the text in `description_clean_2`. Remove all the digits from `description_clean_2` and store the results in another column `description_clean_3` in the `wine_df` data frame.

In [9]:
wine_df['description_clean_3'] = wine_df['description_clean_2'].str.replace(r'\d+', '', regex=True)

### Exercise 1(i) (5 points)

Using the `CounterVectorizer` function from `sklearn.feature_extraction.text`, convert the `description_clean_3` column into matrix of token counts use `max_features=30` to create the input features, called this matrix `X`. Define `y = wine_df['points']`.

In [10]:
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(max_features=30)
X = vec.fit_transform(wine_df['description_clean_3']).toarray()
y = wine_df['points']

### Exercise 1(j) (7 points)

Using 5-fold cross-validation strategy (use `KFold` from `sklearn.linear_model`), train and evaluate the out-of-sample performance of the `LinearRegression` model. Use `mean_squared_error` as measure of performance.

In [11]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_arr = y.values
mse_scores = []
for train_idx, test_idx in kf.split(X):
    model = LinearRegression()
    model.fit(X[train_idx], y_arr[train_idx])
    preds = model.predict(X[test_idx])
    mse_scores.append(mean_squared_error(y_arr[test_idx], preds))
print(np.mean(mse_scores))

8.107463899226861


### Exercise 1(k) (10 points)

Using the same cross-validation strategy from part 1(j), append `country` information to `X` as dummy columns (that is, use `get_dummies` from `pandas` to create dummy columns for `country`). Then, train and evaluate the out-of-fold performance of the `LinearRegression` model. Use `mean_squared_error` as measure of performance. 

In [12]:
country_dummies = pd.get_dummies(wine_df['country'], dummy_na=True).values
X_country = np.hstack([X, country_dummies])

mse_scores_country = []
for train_idx, test_idx in kf.split(X_country):
    model = LinearRegression()
    model.fit(X_country[train_idx], y_arr[train_idx])
    preds = model.predict(X_country[test_idx])
    mse_scores_country.append(mean_squared_error(y_arr[test_idx], preds))
print(np.mean(mse_scores_country))

7.656601800648305


### Exercise 1(l) (3 points)

Based on the results from parts 1(j) and 1(k), what model would use to predict `score`? Please, be specific.

linear regression w/30 token count features plus country\
dummy variables.

### Exercise 1(m) (7 points)

Using the `TfidfVectorizer` function from `sklearn.feature_extraction.text`, convert the `description_clean_3` column into a matrix of TF-IDF features using `max_features=30`. Call this new matrix `X_tfidf`. Using the same 5-fold cross-validation strategy from part 1(j), train and evaluate the out-of-fold performance of the `LinearRegression` model using `X_tfidf` as the input features. Use `mean_squared_error` as the measure of performance.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=30)
X_tfidf = tfidf.fit_transform(wine_df['description_clean_3']).toarray()

mse_scores_tfidf = []
for train_idx, test_idx in kf.split(X_tfidf):
    model = LinearRegression()
    model.fit(X_tfidf[train_idx], y_arr[train_idx])
    preds = model.predict(X_tfidf[test_idx])
    mse_scores_tfidf.append(mean_squared_error(y_arr[test_idx], preds))
print(np.mean(mse_scores_tfidf))

8.065993851530656


### Exercise 1(n) (3 points)

Based on your results from parts 1(j) and 1(m), does replacing the bag-of-words counts with TF-IDF weights improve the out-of-fold performance of the `LinearRegression` model? In your answer, briefly explain **why** TF-IDF might be expected to help (or not) when predicting a wine's `points` score.

tf-idf made a smaller improvement, but not super noticeable.\
in theory it helps bc it downweights words that show up in every entry\
and gives more weight to distinctive ones.

w/only 30 features, both versions are pretty muh the same selection of\
common words.

### Exercise 1(o) (3 points)

Word embeddings such as Word2Vec are described as *static*, while embeddings produced by models like BERT are described as *contextual*. Explain the key difference between these two types of embeddings, and explain why a contextual embedding would likely represent the word `bank` differently in the sentences "I sat by the river bank" and "I deposited a check at the bank", while a static embedding would not.

word2vec gives every word one fixed vector no matter where it shows up. bert builds\ 
the vector from the sentence around it, so the same word can get different\ 
vectors in different contexts.

so for "bank", word2vec has one vector that's a mashup of both meanings, and it uses that exact vector in both sentences. bert sees "river" next to it in one and "deposited a check" in the other, so it spits out two different vectors that actually match the two meanings.